# 🚀 TikTok Sentiment & Topic Intelligence (IndoBERT GPU)
Notebook ini digunakan untuk memproses data komentar TikTok hasil scraping menggunakan model **IndoBERT** di Google Colab (GPU gratis T4).

Hasilnya berupa file CSV yang sudah dilengkapi **Sentimen, Skor, dan Klaster Isu**, yang siap ditampilkan langsung di **Dashboard Web Lokal** kamu.

### Langkah 1: Install Dependensi & Persiapan GPU

In [ ]:
# Install Transformers & PyTorch
!pip install -q transformers torch pandas scikit-learn

import torch
print(f"CUDA Tersedia: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("PERINGATAN: Kamu sedang menggunakan CPU. Aktifkan GPU di menu: Runtime -> Change runtime type -> T4 GPU")

### Langkah 2: Upload File Komentar (misal: `tes_comments.csv`)

In [ ]:
from google.colab import files
import pandas as pd
import os

print("Silakan pilih dan upload file komentar CSV kamu:")
uploaded = files.upload()
uploaded_filename = list(uploaded.keys())[0]

df = pd.read_csv(uploaded_filename)
print(f"Berhasil memuat {len(df)} baris komentar dari {uploaded_filename}!")
display(df.head(3))

### Langkah 3: Muat Model IndoBERT & Jalankan Analisis Sentimen (Cepat di GPU)

In [ ]:
from transformers import pipeline
import time

device = 0 if torch.cuda.is_available() else -1
print("Memuat model IndoBERT (w11wo/indonesian-roberta-base-sentiment-classifier)...")
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier",
    device=device,
    truncation=True,
    max_length=128
)

# Deteksi kolom teks komentar
comment_col = None
for c in ['comment_text', 'text', 'comment', 'isi_komentar']:
    if c in df.columns:
        comment_col = c
        break

texts = df[comment_col].fillna("").astype(str).tolist()
print(f"Memproses {len(texts)} komentar dengan IndoBERT di GPU...")
start_time = time.time()

# Batch inference dengan IndoBERT
batch_size = 64
results = []
for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    preds = sentiment_pipe(batch)
    results.extend(preds)
    if (i // batch_size) % 5 == 0:
        print(f"  Progres: {min(i + batch_size, len(texts))}/{len(texts)} komentar selesai...")

duration = time.time() - start_time
print(f"✅ Selesai dalam {duration:.2f} detik! (Rata-rata: {len(texts)/duration:.1f} komentar/detik)")

# Mapping label IndoBERT ke Bahasa Indonesia standar
label_map = {
    "positive": "Positif",
    "neutral": "Netral",
    "negative": "Negatif",
    "LABEL_0": "Negatif",
    "LABEL_1": "Netral",
    "LABEL_2": "Positif"
}

df['sentiment'] = [label_map.get(r['label'].lower(), r['label'].capitalize()) for r in results]
df['sentiment_score'] = [round(r['score'] if 'pos' in r['label'].lower() else (-r['score'] if 'neg' in r['label'].lower() else 0.0), 3) for r in results]

### Langkah 4: Pemetaan Klaster Isu Hangat & Intisari Pesan

In [ ]:
import re

# Kamus aturan topik spesifik isu daerah & medsos
def classify_topic(text):
    t = text.lower()
    if any(k in t for k in ['bkk', 'pd bkk', 'uang nasabah', 'tabungan', 'korupsi', 'cairkan', 'dana']):
        return 'Keuangan & Kasus BKK'
    if any(k in t for k in ['jalan', 'aspal', 'rusak', 'lobang', 'lubang', 'talut', 'cor', 'proyek', 'jembatan', 'lampu']):
        return 'Infrastruktur & Jalan'
    if any(k in t for k in ['rapor', 'rapot', 'sekolah', 'sdn', 'guru', 'murid', 'libur', 'kelas', 'anak sekolah']):
        return 'Pendidikan & Anak'
    if any(k in t for k in ['konser', 'ndx', 'stadion', 'trikoyo', 'pramuka', 'estafet', 'balap', 'sepeda', 'senam', 'karnaval']):
        return 'Event, Seni & Olahraga'
    if any(k in t for k in ['ktp', 'layanan', 'dinas', 'bupati', 'kantor', 'bansos', 'bantuan', 'desa']):
        return 'Layanan Publik & Birokrasi'
    if any(k in t for k in ['sehat', 'semangat', 'sukses', 'muda', 'ganteng', 'josjis', 'maturnuwun', 'keren', 'idola', 'aamiin']):
        return 'Apresiasi & Doa Personal'
    return 'Lain-lain'

def extract_keypoint(text, topic):
    words = [w for w in re.sub(r'[^a-zA-Z0-9\s]', '', text).split() if len(w) > 2]
    return ' '.join(words[:5]) if words else topic

df['topic'] = df[comment_col].fillna('').apply(classify_topic)
df['key_point'] = [extract_keypoint(txt, top) for txt, top in zip(df[comment_col].fillna(''), df['topic'])]

print("Distribusi Sentimen:")
print(df['sentiment'].value_counts())
print("\nDistribusi Topik:")
print(df['topic'].value_counts())
display(df[['sentiment', 'topic', 'key_point', comment_col]].head(5))

### Langkah 5: Buat Ringkasan Eksekutif & Unduh Hasil

In [ ]:
total = len(df)
pos_pct = (df['sentiment'] == 'Positif').mean() * 100
neg_pct = (df['sentiment'] == 'Negatif').mean() * 100
neu_pct = (df['sentiment'] == 'Netral').mean() * 100
top_issue = df['topic'].value_counts().index[0]

summary_content = f"""Berdasarkan analisis terhadap {total} komentar publik, sentimen masyarakat menunjukkan {pos_pct:.1f}% Positif, {neu_pct:.1f}% Netral, dan {neg_pct:.1f}% Negatif. Topik yang paling mendominasi pembicaraan adalah '{top_issue}'. Isu-isu negatif utama berkisar pada keluhan infrastruktur jalan dan tuntutan nasabah mengenai kasus BKK, sementara sentimen positif didominasi apresiasi terhadap kehadiran pimpinan dan partisipasi dalam kegiatan kemasyarakatan."""

# Simpan hasil CSV dan Summary TXT
out_csv_name = uploaded_filename.replace('.csv', '_analyzed.csv')
out_txt_name = uploaded_filename.replace('.csv', '_summary.txt')

df.to_csv(out_csv_name, index=False, encoding='utf-8')
with open(out_txt_name, 'w', encoding='utf-8') as f:
    f.write(summary_content)

print(f"✅ Berhasil membuat {out_csv_name} dan {out_txt_name}!")
print("\nMengunduh file secara otomatis ke komputermu...")
files.download(out_csv_name)
files.download(out_txt_name)
print("\n👉 Silakan pindahkan file yang diunduh ke folder scrap-tiktok di komputermu, lalu buka Web Dashboard Lokal (http://localhost:5000)!")